# 🧠 Redes Neuronales y Entrenamiento Base con PyTorch

En este notebook vas a construir tu primera **red neuronal** desde cero con PyTorch.
Vas a entender qué es una neurona, cómo se conectan en capas, cómo "aprende" el modelo
y cómo visualizar el proceso de entrenamiento.

### ¿Qué vamos a cubrir?

| Sección | Concepto |
|---|---|
| 1 | Instalación y verificación de PyTorch |
| 2 | ¿Qué es una neurona artificial? |
| 3 | Funciones de activación |
| 4 | Tu primera red neuronal (MLP) |
| 5 | La función de pérdida |
| 6 | Backpropagation y gradientes |
| 7 | Loop de entrenamiento completo |
| 8 | Visualización del entrenamiento |
| 9 | Experimentar con hiperparámetros |
| 10 | Ejercicio evaluable |

> **Prerequisito:** Haber completado NB1 y NB2 (conceptos de IA y trabajo con datos). No se necesita experiencia con PyTorch.

---
## 1. Instalación y verificación de PyTorch

PyTorch es el framework de deep learning más popular en investigación y cada vez más en producción.

### ¿Por qué PyTorch?

| Ventaja | Descripción |
|---|---|
| **Intuitivo** | Se programa como Python normal (eager mode) |
| **Flexible** | Podés modificar todo en tiempo de ejecución |
| **Debug fácil** | Usás `print()` y `pdb` como siempre |
| **Comunidad** | La mayoría de papers publican código en PyTorch |

```bash
pip install torch matplotlib
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import time

print(f"✅ PyTorch versión : {torch.__version__}")
print()

# Detectar el mejor dispositivo disponible
if torch.cuda.is_available():
    device = torch.device("cuda")
    nombre_gpu = torch.cuda.get_device_name(0)
    print(f"🟢 NVIDIA GPU disponible: {nombre_gpu}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🟢 Apple Silicon GPU (Metal MPS) disponible")
else:
    device = torch.device("cpu")
    print("🔵 Usando CPU (sin GPU disponible)")

print(f"\n📍 Dispositivo seleccionado: {device}")
print("   Este notebook corre perfecto en CPU — la red es pequeña a propósito.")

---
## 2. ¿Qué es una neurona artificial?

Una neurona artificial es la unidad básica de una red neuronal. Hace 3 cosas:

```
1. Recibe entradas (x₁, x₂, x₃, ...)
2. Las multiplica por pesos (w₁, w₂, w₃, ...) y suma un sesgo (b)
3. Aplica una función de activación al resultado
```

### Fórmula de una neurona

$$z = w_1 x_1 + w_2 x_2 + ... + w_n x_n + b$$
$$salida = f(z)$$

Donde $f$ es la **función de activación**.

### Analogía

Pensá en una neurona como una **balanza ponderada**:
- Las entradas son los ingredientes
- Los pesos dicen qué tan importante es cada ingrediente
- La activación decide si el resultado "se enciende" o no

> 💡 **Clave:** Los **pesos** y el **sesgo** son los valores que el modelo aprende durante el entrenamiento.

In [ ]:
# Simulación de UNA neurona artificial

print("🔬 Simulación de una neurona artificial\n")

# Entradas: [horas_estudio, asistencia_normalizada]
x = torch.tensor([5.0, 0.8])   # estudia 5 horas, asistió al 80%

# Pesos: cuánto importa cada entrada (el modelo los aprende)
w = torch.tensor([0.6, 0.4])   # horas de estudio importa más

# Sesgo: ajuste global
b = torch.tensor(-1.0)

# Paso 1: combinación lineal
z = torch.dot(w, x) + b       # z = 0.6*5 + 0.4*0.8 + (-1) = 2.32

# Paso 2: función de activación (ReLU: si z > 0 → z, sino → 0)
salida = torch.relu(z)

print(f"📥 Entradas (x)  : {x.tolist()}")
print(f"⚖️  Pesos (w)     : {w.tolist()}")
print(f"➕ Sesgo (b)      : {b.item()}")
print(f"\n🔢 Combinación lineal: z = {w[0]:.1f}×{x[0]:.1f} + {w[1]:.1f}×{x[1]:.1f} + ({b.item():.1f}) = {z.item():.2f}")
print(f"⚡ Activación ReLU  : salida = max(0, {z.item():.2f}) = {salida.item():.2f}")

print("\n💡 Los pesos y el sesgo son lo que el modelo aprende. Las entradas son los datos.")

---
## 3. Funciones de activación

Sin funciones de activación, una red neuronal de muchas capas sería equivalente a una sola operación lineal.
Las activaciones introducen **no-linealidad**, que es lo que permite a las redes aprender patrones complejos.

### Las más usadas

| Función | Fórmula | Cuándo usarla |
|---|---|---|
| **ReLU** | $\max(0, z)$ | Default para capas ocultas (rápida, efectiva) |
| **Sigmoid** | $\frac{1}{1 + e^{-z}}$ | Salida binaria (probabilidad 0-1) |
| **Tanh** | $\frac{e^z - e^{-z}}{e^z + e^{-z}}$ | Salida entre -1 y 1 |
| **Softmax** | $\frac{e^{z_i}}{\sum e^{z_j}}$ | Clasificación multiclase (probabilidades) |

> 💡 **En la práctica:** usá ReLU para las capas ocultas y elegí la activación de la última capa según el tipo de problema.

In [ ]:
# Visualización de las funciones de activación

z = torch.linspace(-5, 5, 200)

activaciones = [
    {"nombre": "ReLU",    "fn": torch.relu(z),         "color": "#4d96ff"},
    {"nombre": "Sigmoid", "fn": torch.sigmoid(z),      "color": "#ff6b6b"},
    {"nombre": "Tanh",    "fn": torch.tanh(z),         "color": "#6bcb77"},
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, act in zip(axes, activaciones):
    ax.plot(z.numpy(), act["fn"].numpy(), color=act["color"], linewidth=2.5)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.axvline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.set_title(act["nombre"], fontsize=14, fontweight="bold")
    ax.set_xlabel("z (entrada)")
    ax.set_ylabel("f(z) (salida)")
    ax.grid(alpha=0.2)

plt.suptitle("Funciones de activación: transforman la señal de cada neurona",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("📌 ReLU: simple y rápida → default para capas ocultas")
print("📌 Sigmoid: comprime todo entre 0 y 1 → útil para probabilidades")
print("📌 Tanh: comprime entre -1 y 1 → centrado en cero")

---
## 4. Tu primera red neuronal (MLP)

Un **MLP** (Multi-Layer Perceptron) es la red neuronal más simple que tiene capas:

```
Entrada → Capa oculta 1 → Capa oculta 2 → Salida
  (4)        (16)             (8)            (1)
```

Cada conexión entre capas tiene **pesos** que se ajustan durante el entrenamiento.

### Vamos a resolver nuestro problema del NB2

- **Entrada:** 4 features numéricas (horas estudio, asistencia, trabajos, sueño)
- **Salida:** 1 número (nota predicha)
- **Tipo de problema:** Regresión

> 💡 En PyTorch, definimos la red como una **clase** que hereda de `nn.Module`.

In [ ]:
# Paso 1: preparar los datos (mismo dataset sintético que NB2)
np.random.seed(42)
n = 200

horas_estudio       = np.random.uniform(0.5, 15, n)
asistencia_pct      = np.clip(np.random.normal(75, 15, n), 10, 100)
trabajos_entregados = np.random.randint(0, 11, n).astype(float)
horas_sueno         = np.clip(np.random.normal(7, 1.5, n), 3, 12)

nota_final = (
    0.3 * horas_estudio
    + 0.02 * asistencia_pct
    + 0.25 * trabajos_entregados
    + 0.15 * horas_sueno
    + np.random.normal(0, 0.8, n)
)
nota_final = np.clip(nota_final, 0, 10)

# Armar matrices X e y
X_np = np.column_stack([horas_estudio, asistencia_pct, trabajos_entregados, horas_sueno])
y_np = nota_final

# Partición train/val (80/20)
idx = np.random.permutation(n)
corte = int(0.8 * n)
X_train_np, X_val_np = X_np[idx[:corte]], X_np[idx[corte:]]
y_train_np, y_val_np = y_np[idx[:corte]], y_np[idx[corte:]]

# Convertir a tensores de PyTorch
X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.float32).unsqueeze(1)  # (N,) → (N,1)
X_val   = torch.tensor(X_val_np, dtype=torch.float32)
y_val   = torch.tensor(y_val_np, dtype=torch.float32).unsqueeze(1)

print(f"✅ Datos preparados")
print(f"   Train : X={X_train.shape}, y={y_train.shape}")
print(f"   Val   : X={X_val.shape}, y={y_val.shape}")
print(f"\n   Tipo de tensor: {X_train.dtype}")
print(f"   Features     : horas_estudio, asistencia, trabajos, sueño")

In [ ]:
# Paso 2: definir la red neuronal (MLP)

class MiPrimeraRed(nn.Module):
    """
    Red neuronal MLP simple para regresión.
    Arquitectura: 4 → 16 → 8 → 1
    """

    def __init__(self):
        super().__init__()

        # Definir las capas (cada una es un grupo de neuronas conectadas)
        self.capa1 = nn.Linear(4, 16)    # 4 entradas → 16 neuronas
        self.capa2 = nn.Linear(16, 8)    # 16 → 8 neuronas
        self.salida = nn.Linear(8, 1)    # 8 → 1 salida (la nota predicha)

    def forward(self, x):
        # forward() define cómo fluyen los datos por la red
        x = F.relu(self.capa1(x))   # Capa 1 + ReLU
        x = F.relu(self.capa2(x))   # Capa 2 + ReLU
        x = self.salida(x)          # Capa de salida (sin activación = regresión)
        return x

# Crear una instancia de la red
modelo = MiPrimeraRed()

# Contar parámetros
n_params = sum(p.numel() for p in modelo.parameters())
n_entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)

print("🏗️  Arquitectura de la red:\n")
print(modelo)
print(f"\n📊 Parámetros totales: {n_params:,}")
print(f"   Todos entrenables: {n_entrenables:,}")

# Desglose por capa
print(f"\n{'─' * 50}")
print("📋 Desglose por capa:\n")
for nombre, param in modelo.named_parameters():
    print(f"   {nombre:<20} {str(param.shape):<15} ({param.numel():>4} parámetros)")

---
## 5. La función de pérdida

La **función de pérdida** (loss function) mide qué tan lejos están las predicciones del modelo de los valores reales.

### Analogía

Es como un **termómetro de error**: cuanto más alta la pérdida, peor está prediciendo el modelo.

### Funciones de pérdida comunes

| Función | Problema | Fórmula simplificada |
|---|---|---|
| **MSE** (Mean Squared Error) | Regresión | $\frac{1}{N} \sum (y - \hat{y})^2$ |
| **Cross-Entropy** | Clasificación | $-\sum y \log(\hat{y})$ |

Para nuestro problema de regresión (predecir nota) usamos **MSE**.

> 💡 **Objetivo del entrenamiento:** reducir la pérdida lo máximo posible.

In [ ]:
# Demostración: cómo se calcula la pérdida

# Predicción de la red SIN entrenar (pesos aleatorios)
with torch.no_grad():  # no necesitamos gradientes para esta demo
    pred_sin_entrenar = modelo(X_train[:5])

print("🎲 Predicciones SIN entrenar (pesos aleatorios):\n")
print(f"{'Real':<12} {'Predicción':<12} {'Error':<12} {'Error²'}")
print("─" * 50)

for real, pred in zip(y_train[:5], pred_sin_entrenar):
    error = (real.item() - pred.item())
    error2 = error ** 2
    print(f"{real.item():<12.2f} {pred.item():<12.2f} {error:<12.2f} {error2:.2f}")

# Calcular MSE
criterio = nn.MSELoss()
loss = criterio(pred_sin_entrenar, y_train[:5])

print(f"\n📊 MSE (pérdida) = {loss.item():.4f}")
print(f"\n💡 La pérdida es alta porque los pesos son aleatorios.")
print(f"   El entrenamiento va a ajustar los pesos para reducir esta pérdida.")

---
## 6. Backpropagation y gradientes

¿Cómo sabe el modelo **en qué dirección** ajustar los pesos para reducir la pérdida?

Usa **gradientes** — la derivada de la pérdida respecto a cada peso.

### El proceso

```
1. Forward pass  → Calcular predicción
2. Calcular loss  → ¿Qué tan mal predijo?
3. Backward pass  → Calcular gradientes (¿cómo cambiar cada peso?)
4. Actualizar pesos → Mover pesos en la dirección que reduce la pérdida
```

### Analogía: bajar una montaña con los ojos vendados

- Estás parado en una montaña (la pérdida es alta)
- No ves, pero podés sentir la pendiente bajo tus pies (el gradiente)
- Das un paso en la dirección más empinada hacia abajo
- Repetís hasta llegar al valle (pérdida mínima)

> 💡 Esto es exactamente el **Gradient Descent** (descenso por gradiente).

In [ ]:
# Demostración paso a paso de forward + backward

print("📐 Forward + Backward paso a paso\n")

# Tomar un mini-batch de 5 ejemplos
xb = X_train[:5]
yb = y_train[:5]

# PASO 1: Forward pass
pred = modelo(xb)
print(f"1️⃣  Forward pass → predicciones: {pred.detach().flatten().tolist()[:3]}...")

# PASO 2: Calcular pérdida
loss = criterio(pred, yb)
print(f"2️⃣  Pérdida (MSE): {loss.item():.4f}")

# PASO 3: Backward pass (calcular gradientes)
modelo.zero_grad()  # limpiar gradientes anteriores
loss.backward()     # calcular gradientes automáticamente

print(f"3️⃣  Backward pass → gradientes calculados")

# Mostrar los gradientes de la primera capa
grad_capa1 = modelo.capa1.weight.grad
print(f"\n📊 Gradientes de la capa 1:")
print(f"   Forma: {grad_capa1.shape}")
print(f"   Promedio absoluto: {grad_capa1.abs().mean().item():.6f}")
print(f"   Máximo absoluto:   {grad_capa1.abs().max().item():.6f}")

print(f"\n💡 Estos gradientes le dicen al optimizador cómo ajustar cada peso.")

---
## 7. Loop de entrenamiento completo

Ahora juntamos todo en el **loop de entrenamiento**: el corazón de cualquier proyecto de deep learning.

### Estructura del loop

```python
for epoca in range(N_EPOCAS):
    # 1. Forward: calcular predicciones
    predicciones = modelo(X_train)

    # 2. Pérdida: medir el error
    loss = criterio(predicciones, y_train)

    # 3. Backward: calcular gradientes
    optimizador.zero_grad()
    loss.backward()

    # 4. Actualizar: ajustar pesos
    optimizador.step()
```

### ¿Qué es una época?

Una **época** = una pasada completa por todos los datos de entrenamiento.
Típicamente necesitamos muchas épocas para que el modelo converja.

> 💡 **Adam** es el optimizador más usado hoy: es rápido y funciona bien sin mucho tuning.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  LOOP DE ENTRENAMIENTO COMPLETO
# ══════════════════════════════════════════════════════════════

# Hiperparámetros
N_EPOCAS       = 300      # cuántas veces pasamos por todos los datos
LEARNING_RATE  = 0.01     # qué tan grande es cada paso de ajuste

# Crear modelo fresco (pesos aleatorios)
modelo = MiPrimeraRed()

# Criterio de pérdida y optimizador
criterio    = nn.MSELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=LEARNING_RATE)

# Historiales para graficar
historial_loss_train = []
historial_loss_val   = []

print(f"🏋️  Entrenamiento: {N_EPOCAS} épocas, lr={LEARNING_RATE}")
print(f"{'─' * 55}")

t_inicio = time.time()

for epoca in range(N_EPOCAS):

    # ── Train ──────────────────────────────────────────────
    modelo.train()                        # modo entrenamiento
    pred_train = modelo(X_train)          # forward
    loss_train = criterio(pred_train, y_train)  # pérdida

    optimizador.zero_grad()               # limpiar gradientes
    loss_train.backward()                 # calcular gradientes
    optimizador.step()                    # actualizar pesos

    # ── Validation (sin gradientes) ────────────────────────
    modelo.eval()                         # modo evaluación
    with torch.no_grad():                 # no calcular gradientes
        pred_val = modelo(X_val)
        loss_val = criterio(pred_val, y_val)

    # Guardar historial
    historial_loss_train.append(loss_train.item())
    historial_loss_val.append(loss_val.item())

    # Imprimir progreso cada 50 épocas
    if (epoca + 1) % 50 == 0 or epoca == 0:
        print(f"  Época {epoca+1:>4}/{N_EPOCAS}  │  "
              f"Loss train: {loss_train.item():.4f}  │  "
              f"Loss val: {loss_val.item():.4f}")

t_total = time.time() - t_inicio

print(f"{'─' * 55}")
print(f"\n✅ Entrenamiento completado en {t_total:.1f} segundos")
print(f"   Loss final train: {historial_loss_train[-1]:.4f}")
print(f"   Loss final val  : {historial_loss_val[-1]:.4f}")

---
## 8. Visualización del entrenamiento

La curva de pérdida es la **herramienta de diagnóstico más importante** en deep learning.

### ¿Qué buscamos?

| Patrón | Significado | Acción |
|---|---|---|
| Train y Val bajan juntos | ✅ El modelo está aprendiendo bien | Seguir entrenando |
| Train baja, Val sube | ⚠️ Overfitting | Parar antes, regularizar, más datos |
| Ambos estancados alto | ❌ Underfitting | Modelo más grande, más épocas, mejor lr |
| Loss oscila mucho | ⚠️ Learning rate muy alto | Reducir lr |

In [ ]:
# Gráfico de pérdida: la herramienta de diagnóstico #1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Gráfico 1: Pérdida completa ──────────────────────────
axes[0].plot(historial_loss_train, label="Train", color="#4d96ff", linewidth=1.5)
axes[0].plot(historial_loss_val, label="Validation", color="#ff6b6b", linewidth=1.5)
axes[0].set_xlabel("Época", fontsize=11)
axes[0].set_ylabel("Pérdida (MSE)", fontsize=11)
axes[0].set_title("Curva de pérdida completa", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.2)

# ── Gráfico 2: Últimas 200 épocas (zoom) ─────────────────
inicio_zoom = max(0, len(historial_loss_train) - 200)
axes[1].plot(range(inicio_zoom, N_EPOCAS), historial_loss_train[inicio_zoom:],
             label="Train", color="#4d96ff", linewidth=1.5)
axes[1].plot(range(inicio_zoom, N_EPOCAS), historial_loss_val[inicio_zoom:],
             label="Validation", color="#ff6b6b", linewidth=1.5)
axes[1].set_xlabel("Época", fontsize=11)
axes[1].set_ylabel("Pérdida (MSE)", fontsize=11)
axes[1].set_title("Zoom: últimas épocas", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.2)

plt.suptitle("Diagnóstico del entrenamiento", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Diagnóstico automático
gap = historial_loss_val[-1] - historial_loss_train[-1]
print(f"\n📊 Diferencia train-val (gap): {gap:.4f}")
if gap < 0.1:
    print("✅ Gap pequeño → el modelo generaliza bien.")
elif gap < 0.5:
    print("🟡 Gap moderado → vigilar overfitting si seguimos entrenando.")
else:
    print("⚠️  Gap grande → posible overfitting. Considerar regularización.")

In [ ]:
# Gráfico: predicciones vs. valores reales (después de entrenar)

modelo.eval()
with torch.no_grad():
    pred_final_train = modelo(X_train).numpy()
    pred_final_val   = modelo(X_val).numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_real, y_pred, titulo, color in [
    (axes[0], y_train.numpy(), pred_final_train, "Train", "#4d96ff"),
    (axes[1], y_val.numpy(),   pred_final_val,   "Validation", "#ff6b6b"),
]:
    ax.scatter(y_real, y_pred, alpha=0.5, color=color, edgecolor="gray", s=40)
    ax.plot([0, 10], [0, 10], "--", color="red", linewidth=1.5, label="Predicción perfecta")
    ax.set_xlabel("Nota real", fontsize=11)
    ax.set_ylabel("Nota predicha", fontsize=11)
    ax.set_title(titulo, fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_xlim(0, 10.5)
    ax.set_ylim(0, 10.5)
    ax.set_aspect("equal")

plt.suptitle("Red neuronal: Real vs. Predicho", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 9. Experimentar con hiperparámetros

Los **hiperparámetros** son las decisiones que tomamos NOSOTROS antes de entrenar (no las aprende el modelo).

### Los más importantes para empezar

| Hiperparámetro | Qué controla | Rango típico |
|---|---|---|
| **Learning rate** | Tamaño del paso de ajuste | 0.0001 — 0.1 |
| **Épocas** | Cuántas pasadas por los datos | 50 — 10,000 |
| **Neuronas por capa** | Capacidad del modelo | 8 — 512 |
| **Número de capas** | Profundidad del modelo | 1 — 10 |

> 💡 **Regla práctica:** empezá con valores razonables y cambiá UNO a la vez para entender su efecto.

In [ ]:
# 🔬 Experimento: ¿qué pasa al cambiar el learning rate?

learning_rates = [0.0001, 0.001, 0.01, 0.1]

fig, ax = plt.subplots(figsize=(10, 5))
colores = ["#4d96ff", "#6bcb77", "#ffd93d", "#ff6b6b"]

print("🔬 Experimento: efecto del learning rate\n")

for lr, color in zip(learning_rates, colores):
    # Modelo fresco
    torch.manual_seed(42)
    modelo_exp = MiPrimeraRed()
    opt_exp = torch.optim.Adam(modelo_exp.parameters(), lr=lr)
    crit_exp = nn.MSELoss()

    losses = []
    for epoca in range(200):
        modelo_exp.train()
        pred = modelo_exp(X_train)
        loss = crit_exp(pred, y_train)
        opt_exp.zero_grad()
        loss.backward()
        opt_exp.step()
        losses.append(loss.item())

    ax.plot(losses, label=f"lr={lr}", color=color, linewidth=1.8)
    print(f"  lr={lr:<8} → Loss final: {losses[-1]:.4f}")

ax.set_xlabel("Época", fontsize=11)
ax.set_ylabel("Pérdida (MSE)", fontsize=11)
ax.set_title("Efecto del Learning Rate en el entrenamiento", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.2)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

print("\n📌 Muy bajo: aprende lento. Muy alto: puede no converger.")
print("📌 La clave es encontrar el balance.")

In [ ]:
# 🔬 Experimento: ¿qué pasa al cambiar el tamaño de la red?

class RedChica(nn.Module):
    def __init__(self):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(4, 4), nn.ReLU(),
            nn.Linear(4, 1)
        )
    def forward(self, x): return self.capas(x)

class RedMediana(nn.Module):
    def __init__(self):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(4, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1)
        )
    def forward(self, x): return self.capas(x)

class RedGrande(nn.Module):
    def __init__(self):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(4, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x): return self.capas(x)

redes = [
    {"nombre": "Chica (4→4→1)",       "clase": RedChica,   "color": "#4d96ff"},
    {"nombre": "Mediana (4→16→8→1)",   "clase": RedMediana, "color": "#6bcb77"},
    {"nombre": "Grande (4→64→32→16→1)", "clase": RedGrande,  "color": "#ff6b6b"},
]

fig, ax = plt.subplots(figsize=(10, 5))

print("🔬 Experimento: efecto del tamaño de la red\n")

for red in redes:
    torch.manual_seed(42)
    m = red["clase"]()
    n_p = sum(p.numel() for p in m.parameters())
    opt = torch.optim.Adam(m.parameters(), lr=0.01)
    crit = nn.MSELoss()

    losses_val = []
    for epoca in range(300):
        m.train()
        loss = crit(m(X_train), y_train)
        opt.zero_grad(); loss.backward(); opt.step()

        m.eval()
        with torch.no_grad():
            lv = crit(m(X_val), y_val).item()
        losses_val.append(lv)

    ax.plot(losses_val, label=f"{red['nombre']} ({n_p} params)",
            color=red["color"], linewidth=1.8)
    print(f"  {red['nombre']:<30} {n_p:>5} params → Loss val final: {losses_val[-1]:.4f}")

ax.set_xlabel("Época", fontsize=11)
ax.set_ylabel("Pérdida Validation", fontsize=11)
ax.set_title("Efecto del tamaño de la red en la pérdida de validación",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.2)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

print("\n📌 Más grande no siempre es mejor — depende de la complejidad del problema.")

---
## 10. Ejercicio evaluable: tu experimento con redes neuronales

### Tu entregable

Completá las celdas a continuación. El objetivo es que demuestres que entendés cómo funciona el entrenamiento.

### Criterio de aprobación

- ✅ Definir una red neuronal con al menos 2 capas ocultas
- ✅ Entrenar con un loop completo y guardar el historial de pérdida
- ✅ Graficar la curva de pérdida train vs. val
- ✅ Responder las preguntas de análisis

> 💡 **Tip:** no busques la red perfecta. Lo importante es que el entrenamiento funcione y que puedas interpretarlo.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  EJERCICIO EVALUABLE — Parte 1: Definir tu red
# ══════════════════════════════════════════════════════════════

class MiRedPersonalizada(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: definir al menos 2 capas ocultas
        # Ejemplo: self.capa1 = nn.Linear(..., ...)
        pass

    def forward(self, x):
        # TODO: definir el flujo de datos
        # Ejemplo: x = F.relu(self.capa1(x))
        return x

# Verificar que la red funciona
try:
    mi_modelo = MiRedPersonalizada()
    test_out = mi_modelo(X_train[:1])
    n_p = sum(p.numel() for p in mi_modelo.parameters())
    print(f"✅ Red creada correctamente")
    print(f"   Salida de prueba: {test_out.item():.4f}")
    print(f"   Parámetros: {n_p:,}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("   Revisá la definición de __init__ y forward")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  EJERCICIO EVALUABLE — Parte 2: Entrenar y graficar
# ══════════════════════════════════════════════════════════════

# TODO: Elegir hiperparámetros
MI_LEARNING_RATE = ...   # Ej: 0.01
MIS_EPOCAS       = ...   # Ej: 300

# TODO: Crear optimizador y criterio
# mi_optimizador = ...
# mi_criterio    = ...

# TODO: Escribir el loop de entrenamiento
# Guardá historial de loss train y val para graficar

# TODO: Graficar curvas de pérdida train vs. val

print("📌 Completá este código y ejecutá la celda.")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  EJERCICIO EVALUABLE — Parte 3: Análisis
# ══════════════════════════════════════════════════════════════

# Respondé estas preguntas basándote en tus resultados:

respuestas = {
    # 1. ¿Tu modelo muestra señales de overfitting? ¿Cómo lo sabés?
    "overfitting": "",  # TODO

    # 2. ¿Qué learning rate usaste y por qué?
    "learning_rate": "",  # TODO

    # 3. ¿Cuántas épocas entrenaste? ¿Fue suficiente?
    "epocas": "",  # TODO

    # 4. ¿Qué cambiarías para mejorar el modelo?
    "mejora": "",  # TODO
}

# Validación
vacias = [k for k, v in respuestas.items() if v.strip() == ""]
if vacias:
    print(f"⚠️  Faltan {len(vacias)} respuestas: {vacias}")
else:
    print("✅ ¡Análisis completo!\n")
    for pregunta, resp in respuestas.items():
        print(f"  📌 {pregunta}: {resp}")
    print("\n🎉 Entregable listo. Guardá este notebook y envialo.")

---
## Resumen del notebook

| Concepto | Lo que aprendiste |
|---|---|
| Neurona artificial | Pesos × entradas + sesgo → activación |
| Funciones de activación | ReLU, Sigmoid, Tanh — introducen no-linealidad |
| MLP | Red de capas conectadas — la arquitectura más simple |
| Función de pérdida | MSE para regresión — mide qué tan mal predice |
| Backpropagation | Cálculo automático de gradientes |
| Loop de entrenamiento | Forward → Loss → Backward → Update |
| Curvas de pérdida | Herramienta #1 de diagnóstico |
| Hiperparámetros | Learning rate, épocas, tamaño de red |

### Próximo notebook

En **NB4** vamos a escalar todo esto y construir un **mini-LLM desde cero**: tokenización, embeddings, self-attention y generación de texto.

> 📌 **Recordá:** una red neuronal no es magia — es multiplicar matrices, medir error y ajustar pesos. Repetir hasta converger.